# Redis Demo
## Beispiel: Bundesliga-Liveticker

Wir bauen einen **Bundesliga-Liveticker** für die Webseite und App eines großen Sportportals.

Das Portal hat ein Performance-Problem:
- Vereinsdaten, Spielpläne und Tabellen liegen in einer relationalen Datenbank
- Jeder Aufruf eines Spiels erzeugt einen teuren DB-Query (Joins über `matches`, `teams`, `players`, `events`, ...)
- An einem Spieltag-Samstag um 15:30 Uhr klicken hunderttausende User gleichzeitig F5 - die DB bricht zusammen

Lösung: **Redis als Caching-Layer** vor der Datenbank.

Zusätzlich brauchen wir:
- **Live-Tabelle** mit blitzschnellem Update bei jedem Tor
- **Live-Spielstand** mit kurzer Ablaufzeit (TTL)
- **User-Sessions** für eingeloggte Fans
- **Zuschauer-Counter** pro Spiel ("Wieviele schauen gerade Bayern - Dortmund mit?")
- **Rate-Limiting** für die API
- **Replication** für Ausfallsicherheit bei Spitzenspielen

## Schritt 1: Setup für Google Colab

Wir installieren Redis direkt in der Colab-VM und starten es als Hintergrundprozess. Das funktioniert, weil Colab eine vollständige Linux-Umgebung mit `sudo`-Zugriff ist.

In [1]:
# Redis Server im Colab installieren (einmalig pro Session)
!apt-get install -y redis-server > /dev/null
!redis-server --daemonize yes --save "" --appendonly no
print("Redis-Server läuft auf localhost:6379")

Redis-Server läuft auf localhost:6379


In [2]:
# Python-Client installieren
!pip install -q redis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.8/409.8 kB 20.2 MB/s eta 0:00:00


In [3]:
import redis
import time
import json
import subprocess
import random
import hashlib
import secrets
from collections import Counter
from datetime import datetime

## Schritt 2: Verbindung zu Redis aufbauen

`redis.Redis(...)` erzeugt einen Client. `decode_responses=True` sorgt dafür, dass Antworten als Strings zurückkommen statt als Bytes.

In [4]:
r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)

# Ping testet ob Redis erreichbar ist
print("PING:", r.ping())

# Sauber starten: alle Keys in der aktuellen DB löschen
r.flushdb()
print("DB resettet.")

PING: True
DB resettet.


## Schritt 3: Unsere "Datenbank": Bundesliga-Saison 2025/26

Wir simulieren den Stand **nach dem 12. Spieltag**. Es laufen gerade die Partien des 13. Spieltags.

In [5]:
# Bundesliga-Tabelle nach dem 12. Spieltag der Saison 2025/26
# (plausible Werte, nicht die exakten echten Daten)

TEAMS = {
    "FCB": {"name": "FC Bayern München",         "stadium": "Allianz Arena",     "founded": 1900, "city": "München"},
    "BVB": {"name": "Borussia Dortmund",         "stadium": "Signal Iduna Park", "founded": 1909, "city": "Dortmund"},
    "RBL": {"name": "RB Leipzig",                "stadium": "Red Bull Arena",    "founded": 2009, "city": "Leipzig"},
    "B04": {"name": "Bayer 04 Leverkusen",       "stadium": "BayArena",          "founded": 1904, "city": "Leverkusen"},
    "VFB": {"name": "VfB Stuttgart",             "stadium": "MHPArena",          "founded": 1893, "city": "Stuttgart"},
    "SGE": {"name": "Eintracht Frankfurt",       "stadium": "Deutsche Bank Park","founded": 1899, "city": "Frankfurt"},
    "TSG": {"name": "TSG Hoffenheim",            "stadium": "PreZero Arena",     "founded": 1899, "city": "Sinsheim"},
    "SCF": {"name": "SC Freiburg",               "stadium": "Europa-Park Stadion","founded": 1904,"city": "Freiburg"},
    "FCA": {"name": "FC Augsburg",               "stadium": "WWK Arena",         "founded": 1907, "city": "Augsburg"},
    "M05": {"name": "1. FSV Mainz 05",           "stadium": "Mewa Arena",        "founded": 1905, "city": "Mainz"},
    "HSV": {"name": "Hamburger SV",              "stadium": "Volksparkstadion",  "founded": 1887, "city": "Hamburg"},
    "FCU": {"name": "1. FC Union Berlin",        "stadium": "Alte Försterei",    "founded": 1966, "city": "Berlin"},
    "BMG": {"name": "Borussia Mönchengladbach",  "stadium": "Borussia-Park",     "founded": 1900, "city": "Mönchengladbach"},
    "KOE": {"name": "1. FC Köln",                "stadium": "RheinEnergieStadion","founded": 1948,"city": "Köln"},
    "SVW": {"name": "SV Werder Bremen",          "stadium": "Weserstadion",      "founded": 1899, "city": "Bremen"},
    "WOB": {"name": "VfL Wolfsburg",             "stadium": "Volkswagen Arena",  "founded": 1945, "city": "Wolfsburg"},
    "FCH": {"name": "1. FC Heidenheim",          "stadium": "Voith-Arena",       "founded": 1846, "city": "Heidenheim"},
    "STP": {"name": "FC St. Pauli",              "stadium": "Millerntor-Stadion","founded": 1910, "city": "Hamburg"},
}

# Tabelle nach 12 Spieltagen: (Punkte, Tore, Gegentore)
STANDINGS = {
    "FCB": (31, 35, 9),
    "BVB": (25, 26, 14),
    "RBL": (23, 22, 12),
    "B04": (22, 24, 13),
    "VFB": (22, 21, 15),
    "SGE": (20, 23, 17),
    "TSG": (18, 19, 18),
    "SCF": (17, 16, 16),
    "FCA": (15, 14, 17),
    "M05": (14, 15, 19),
    "HSV": (14, 13, 17),
    "FCU": (13, 12, 17),
    "BMG": (12, 16, 22),
    "KOE": (11, 13, 21),
    "SVW": (11, 14, 23),
    "WOB": (9,  10, 22),
    "FCH": (8,  11, 25),
    "STP": (7,  9,  26),
}

# Spielplan des 13. Spieltags (heutiger Spieltag)
MATCHDAY_13 = [
    {"id": "match:1301", "home": "FCB", "away": "BVB", "kickoff": "Sa 18:30", "highlight": True},
    {"id": "match:1302", "home": "RBL", "away": "B04", "kickoff": "Sa 15:30", "highlight": True},
    {"id": "match:1303", "home": "VFB", "away": "SGE", "kickoff": "Sa 15:30", "highlight": False},
    {"id": "match:1304", "home": "TSG", "away": "SCF", "kickoff": "Sa 15:30", "highlight": False},
    {"id": "match:1305", "home": "FCA", "away": "M05", "kickoff": "Sa 15:30", "highlight": False},
    {"id": "match:1306", "home": "HSV", "away": "FCU", "kickoff": "Sa 15:30", "highlight": False},
    {"id": "match:1307", "home": "BMG", "away": "KOE", "kickoff": "So 15:30", "highlight": False},
    {"id": "match:1308", "home": "SVW", "away": "WOB", "kickoff": "So 17:30", "highlight": False},
    {"id": "match:1309", "home": "FCH", "away": "STP", "kickoff": "Fr 20:30", "highlight": False},
]

print(f"{len(TEAMS)} Vereine in der Bundesliga 2025/26")
print(f"Tabellenführer: {TEAMS['FCB']['name']} mit {STANDINGS['FCB'][0]} Punkten")
print(f"Schlusslicht:   {TEAMS['STP']['name']} mit {STANDINGS['STP'][0]} Punkten")
print(f"\nTop-Spiel heute: {TEAMS['FCB']['name']} - {TEAMS['BVB']['name']}")

18 Vereine in der Bundesliga 2025/26
Tabellenführer: FC Bayern München mit 31 Punkten
Schlusslicht:   FC St. Pauli mit 7 Punkten

Top-Spiel heute: FC Bayern München - Borussia Dortmund


---
## Schritt 4: Konzept

Redis = **RE**mote **DI**ctionary **S**erver. Ein In-Memory Key-Value Store.

### Frage an euch
Wir haben in der letzten Demo den Decathlon-Produktkatalog in MongoDB gespeichert.
Würdet ihr den **kompletten Bundesliga-Datenbestand** (alle Spiele aller Saisons, alle Spieler, alle Statistiken) jetzt in Redis ablegen? Warum (nicht)?

### Lösung
**Nein**, normalerweise nicht. Gründe:
- Redis hält Daten primär im RAM → teuer für große Datenmengen (z.B. ganze Spielerhistorien)
- Keine reichhaltigen Queries (z.B. *alle Tore von Harry Kane im Auswärtsspiel gegen Mannschaften aus dem Ruhrgebiet*)
- Persistenz ist möglich, aber Redis ist nicht als primäre Datenbank konzipiert

**Aber:** Als *Cache-Schicht* vor der primären DB ist Redis ideal. Häufig abgefragte Spiele landen für ein paar Sekunden bis Minuten in Redis, die historischen Detaildaten bleiben in der DB.

---
## Schritt 5: SET & GET: Die elementaren Operationen

Alles in Redis dreht sich um Keys. Ein Key ist ein String, der Wert kann verschiedene Datentypen haben.

### Einfacher String-Wert

In [6]:
r.set("greeting", "Willkommen beim Bundesliga-Liveticker!")
print(r.get("greeting"))

Willkommen beim Bundesliga-Liveticker!


### Namespacing über Doppelpunkte
Konvention in Redis: Keys mit `:` strukturieren, z.B. `team:FCB:points`, `match:1301:score`.
Das ist keine technische Hierarchie, aber hilft beim Aufräumen und Suchen.

In [7]:
r.set("team:FCB:name", "FC Bayern München")
r.set("team:FCB:stadium", "Allianz Arena")
r.set("team:FCB:points", 31)

print(r.get("team:FCB:name"))
print(r.get("team:FCB:stadium"))
print(r.get("team:FCB:points"))   # kommt als String zurück!

FC Bayern München
Allianz Arena
31


### Mehrere Keys auf einmal: MSET / MGET

In [8]:
r.mset({
    "team:BVB:name": "Borussia Dortmund",
    "team:BVB:stadium": "Signal Iduna Park",
    "team:BVB:points": 25
})

namen = r.mget("team:FCB:name", "team:BVB:name", "team:RBL:name")
for n in namen:
    print("-", n)   # team:RBL:name existiert noch nicht -> None

- FC Bayern München
- Borussia Dortmund
- None


### Existenz prüfen und löschen

In [9]:
print("team:FCB:name existiert?", r.exists("team:FCB:name"))
print("team:XYZ:name existiert?", r.exists("team:XYZ:name"))

r.delete("team:BVB:points")
print("Nach DELETE - team:BVB:points:", r.get("team:BVB:points"))

team:FCB:name existiert? 1
team:XYZ:name existiert? 0
Nach DELETE - team:BVB:points: None


### Keys auflisten (Achtung: Anti-Pattern in Produktion!)
`KEYS pattern` blockiert Redis bei großen Datasets. In Produktion stattdessen `SCAN` verwenden.

In [11]:
print("Alle team:FCB:* Keys:")
for k in r.keys("team:FCB:*"):
    print(" ", k, "=", r.get(k))

Alle team:FCB:* Keys:
  team:FCB:stadium = Allianz Arena
  team:FCB:points = 31
  team:FCB:name = FC Bayern München


---
## Schritt 6: Caching-Layer: Das Cache-Aside-Pattern

Das ist der Kern der Demo: Wir simulieren eine **langsame Datenbank** und legen einen Redis-Cache davor.

### Cache-Aside-Pattern
```
1. User öffnet Live-Seite zum Spiel Bayern - Dortmund
2. Anwendung schaut zuerst in Redis
   - HIT  -> fertig, Antwort kommt aus dem Cache (unter 1ms)
   - MISS -> fragt die langsame DB
            schreibt Ergebnis in Redis (TTL setzen!)
            antwortet dem User
```

### Die langsame Datenbank simulieren
Jeder DB-Query hat künstliche Latenz, wie es bei einer komplexen Query mit JOINs der Fall wäre.

In [15]:
def get_match_from_db(match_id):
    """
    Simuliert einen langsamen DB-Query mit 300ms Latenz.
    Würde in echt mehrere JOINs machen (matches, teams, events, players, ...).
    """
    time.sleep(0.3)
    for m in MATCHDAY_13:
        if m["id"] == match_id:
            # Wir bauen ein "fettes" Antwort-Objekt mit allen Infos für die Live-Seite
            return {
                "id": m["id"],
                "home_team": TEAMS[m["home"]]["name"],
                "away_team": TEAMS[m["away"]]["name"],
                "stadium": TEAMS[m["home"]]["stadium"],
                "kickoff": m["kickoff"],
                "is_highlight": m["highlight"]
            }
    return None

# Test: Spiel direkt aus der DB holen
t0 = time.perf_counter()
spiel = get_match_from_db("match:1301")
dt = (time.perf_counter() - t0) * 1000
print(f"DB-Query dauerte {dt:.1f} ms")
print(spiel)

DB-Query dauerte 300.4 ms
{'id': 'match:1301', 'home_team': 'FC Bayern München', 'away_team': 'Borussia Dortmund', 'stadium': 'Allianz Arena', 'kickoff': 'Sa 18:30', 'is_highlight': True}


### Cache-Aside-Funktion

In [16]:
CACHE_TTL_SECONDS = 60

def get_match_cached(match_id):
    """
    Liefert das Spiel zurück und gibt an, ob es aus dem Cache oder aus der DB kam.
    Returns: (match, source) mit source in {'CACHE', 'DB'}
    """
    cache_key = f"cache:match:{match_id}"

    # 1. Cache versuchen
    cached = r.get(cache_key)
    if cached is not None:
        return json.loads(cached), "CACHE"

    # 2. Cache MISS -> DB fragen
    match = get_match_from_db(match_id)
    if match is None:
        return None, "DB"

    # 3. Ergebnis in den Cache schreiben (mit TTL!)
    r.set(cache_key, json.dumps(match, ensure_ascii=False), ex=CACHE_TTL_SECONDS)
    return match, "DB"

### Erster Aufruf vs. zweiter Aufruf
Beim ersten Aufruf ist der Cache leer → DB-Query (langsam).
Beim zweiten Aufruf liegt das Spiel im Cache → blitzschnell.

In [17]:
# Erster Aufruf: Cache MISS
t0 = time.perf_counter()
match, source = get_match_cached("match:1301")
dt = (time.perf_counter() - t0) * 1000
print(f"1. Aufruf:  {dt:7.2f} ms   ({source})   -> {match['home_team']} vs {match['away_team']}")

# Zweiter Aufruf: Cache HIT
t0 = time.perf_counter()
match, source = get_match_cached("match:1301")
dt = (time.perf_counter() - t0) * 1000
print(f"2. Aufruf:  {dt:7.2f} ms   ({source})   -> {match['home_team']} vs {match['away_team']}")

# Dritter Aufruf: Cache HIT
t0 = time.perf_counter()
match, source = get_match_cached("match:1301")
dt = (time.perf_counter() - t0) * 1000
print(f"3. Aufruf:  {dt:7.2f} ms   ({source})   -> {match['home_team']} vs {match['away_team']}")

1. Aufruf:   304.09 ms   (DB)   -> FC Bayern München vs Borussia Dortmund
2. Aufruf:     0.73 ms   (CACHE)   -> FC Bayern München vs Borussia Dortmund
3. Aufruf:     0.35 ms   (CACHE)   -> FC Bayern München vs Borussia Dortmund


### Realistisches Szenario: Topspiel zieht die Aufmerksamkeit

Samstag 18:30: Bayern gegen Dortmund. Die Hälfte der User klickt nur dieses eine Spiel an. Andere Spiele werden auch aufgerufen, aber viel seltener. **Power-Law-Verteilung**.

In [ ]:
r.flushdb()  # Cache leeren für saubere Messung

# 500 Requests, stark gewichtet auf das Topspiel
match_ids = [m["id"] for m in MATCHDAY_13]
# Topspiel Bayern-BVB bekommt 20x mehr Aufrufe als andere Spiele
weights = [20 if m["highlight"] else 1 for m in MATCHDAY_13]

hits, misses = 0, 0
t0 = time.perf_counter()
for _ in range(500):
    mid = random.choices(match_ids, weights=weights, k=1)[0] # 1 Match_ID
    _, source = get_match_cached(mid)
    if source == "CACHE":
        hits += 1
    else:
        misses += 1
total_ms = (time.perf_counter() - t0) * 1000

print(f"500 Requests in {total_ms:.0f} ms")
print(f"Cache-Hits:   {hits:3d}  ({hits/5:.1f}%)")
print(f"Cache-Misses: {misses:3d}  ({misses/5:.1f}%)")
print(f"\nOhne Cache wären das mindestens {500 * 300:.0f} ms gewesen.")

500 Requests in 2810 ms
Cache-Hits:   491  (98.2%)
Cache-Misses:   9  (1.8%)

Ohne Cache wären das mindestens 150000 ms gewesen.


### Cache invalidieren: Bayern schießt ein Tor!

Sobald sich der Spielstand ändert, ist der Cache-Inhalt veraltet. Klassische Strategien:
- **TTL**: Eintrag läuft nach X Sekunden automatisch ab (Konsistenz "irgendwann")
- **Write-Through**: Bei jedem Update auch den Cache aktualisieren
- **Cache-Invalidation**: Bei jedem Update den Cache-Key löschen

In [18]:
# Vor dem Tor: Spiel ist im Cache
get_match_cached("match:1301")
print("Cache-Key existiert:", r.exists("cache:match:match:1301"))

# Schiedsrichter erkennt das Tor an, Liveticker-Backend invalidiert den Cache:
r.delete("cache:match:match:1301")
print("Nach DELETE:        ", r.exists("cache:match:match:1301"))

# Nächster User-Request lädt frisch aus der DB
_, source = get_match_cached("match:1301")
print(f"Nächster Aufruf:    {source}")

Cache-Key existiert: 1
Nach DELETE:         0
Nächster Aufruf:    DB


---
## Schritt 7: Redis-Datentypen jenseits von Strings

Redis ist mehr als nur ein simpler Key-Value-Store. Es kennt mehrere strukturierte Datentypen.

### Hashes - ein Key, viele Felder
Statt für jedes Feld einen eigenen Key (`team:FCB:name`, `team:FCB:stadium` ...) packen wir alles in einen Hash. Effizienter und atomar updatebar.

In [19]:
# Alte String-Keys aus Sektion 4 aufräumen, damit der Hash-Key 'team:FCB' sauber demonstriert wird
for k in r.keys("team:*:*"):
    r.delete(k)

# Vereinsdaten als Hash speichern
r.hset("team:FCB", mapping={
    "name": "FC Bayern München",
    "stadium": "Allianz Arena",
    "founded": 1900,
    "city": "München",
    "points": 31,
    "goals_for": 35,
    "goals_against": 9
})

# Einzelnes Feld lesen
print("Stadion:", r.hget("team:FCB", "stadium"))

# Alle Felder lesen
print("Alles:  ", r.hgetall("team:FCB"))

# Mehrere Felder gleichzeitig
print("Auswahl:", r.hmget("team:FCB", ["name", "points"]))

Stadion: Allianz Arena
Alles:   {'name': 'FC Bayern München', 'stadium': 'Allianz Arena', 'founded': '1900', 'city': 'München', 'points': '31', 'goals_for': '35', 'goals_against': '9'}
Auswahl: ['FC Bayern München', '31']


**Hashes können auch atomar inkrementiert werden** - super praktisch für Tore!

In [20]:
# Bayern schießt ein Tor:
r.hincrby("team:FCB", "goals_for", 1)
print("Tore Bayern jetzt:", r.hget("team:FCB", "goals_for"))

Tore Bayern jetzt: 36


### Lists - geordnete Sequenzen (FIFO/LIFO)
Use Case: Live-Tor-Ticker - die letzten Events werden chronologisch eingespielt.

In [22]:
r.delete("ticker:match:1301")

# Tor-Events werden eingespielt (LPUSH = vorne anhängen)
r.lpush("ticker:match:1301", "12' Tor! Kane (Bayern) trifft zum 1:0")
r.lpush("ticker:match:1301", "23' Gelbe Karte für Süle (BVB)")
r.lpush("ticker:match:1301", "34' Tor! Guirassy (BVB) gleicht zum 1:1 aus")
r.lpush("ticker:match:1301", "45' Halbzeit")

# Die letzten 10 Events anzeigen (neueste zuerst)
print("Live-Ticker:")
for event in r.lrange("ticker:match:1301", 0, 9):
    print(" ", event)
print(f"\nAnzahl Events: {r.llen('ticker:match:1301')}")

Live-Ticker:
  45' Halbzeit
  34' Tor! Guirassy (BVB) gleicht zum 1:1 aus
  23' Gelbe Karte für Süle (BVB)
  12' Tor! Kane (Bayern) trifft zum 1:0

Anzahl Events: 4


### Sets - ungeordnete Mengen ohne Duplikate
Use Case: *Welche User sind Fans von Bayern?* - ohne Doppelzählung.

In [23]:
r.delete("fans:FCB")

# Mehrere User abonnieren Bayern-Updates
r.sadd("fans:FCB", "user:42", "user:17", "user:99", "user:108")
r.sadd("fans:FCB", "user:42")  # Duplikat, wird ignoriert

print("Anzahl Bayern-Fans:", r.scard("fans:FCB"))
print("Fans:              ", r.smembers("fans:FCB"))
print("Ist user:42 Fan?   ", r.sismember("fans:FCB", "user:42"))

# Set-Operationen: Wer ist Fan von BEIDEN Vereinen?
r.sadd("fans:BVB", "user:42", "user:200", "user:300")
print("Schnittmenge FCB UND BVB:", r.sinter("fans:FCB", "fans:BVB"))

Anzahl Bayern-Fans: 4
Fans:               {'user:108', 'user:42', 'user:17', 'user:99'}
Ist user:42 Fan?    1
Schnittmenge FCB UND BVB: {'user:42'}


### Sorted Sets - DIE Tabelle!

Das ist der Killer-Use-Case: Ein Sorted Set ordnet Elemente nach einem **Score** automatisch. Wenn der Score = Punkte ist, hat man **kostenlos eine Live-Tabelle**, die bei jedem Update sofort konsistent ist.

In [24]:
r.delete("standings")

# Komplette Tabelle als Sorted Set aufbauen
# Wir nutzen Punkte + (Tordifferenz / 1000) als Score, damit die Sortierung exakt der echten Tabelle entspricht
for code_, (pts, gf, ga) in STANDINGS.items():
    gd = gf - ga
    score = pts + gd / 1000   # Punkte primär, Tordifferenz sekundär als Tie-Breaker
    r.zadd("standings", {code_: score})

# Top 5 Vereine (höchster Score zuerst)
print("Aktuelle Tabelle (Top 5):")
print(f"{'Pl.':>3} {'Verein':25s} {'Pkt':>4} {'TD':>4}")
for i, (code_, score) in enumerate(r.zrevrange("standings", 0, 4, withscores=True), 1):
    pts, gf, ga = STANDINGS[code_]
    print(f"{i:3d}. {TEAMS[code_]['name']:25s} {pts:4d} {gf-ga:+4d}")

Aktuelle Tabelle (Top 5):
Pl. Verein                     Pkt   TD
  1. FC Bayern München           31  +26
  2. Borussia Dortmund           25  +12
  3. RB Leipzig                  23  +10
  4. Bayer 04 Leverkusen         22  +11
  5. VfB Stuttgart               22   +6


**Die ganze Magie:** Bei einem Tor ändert sich nur EIN Score - die Tabelle ist sofort neu sortiert. Kein `ORDER BY`, kein Re-Compute, alles in der DB-Engine atomar.

In [25]:
# Beispiel: Bayern gewinnt das Topspiel 3:1 gegen BVB
# -> Bayern +3 Punkte, +2 Tordifferenz
# -> BVB +0 Punkte, -2 Tordifferenz

# Bayern aktualisieren
old_score = r.zscore("standings", "FCB")
new_pts = STANDINGS["FCB"][0] + 3
new_gd = (STANDINGS["FCB"][1] - STANDINGS["FCB"][2]) + 2
r.zadd("standings", {"FCB": new_pts + new_gd / 1000})

# Den Tabellenplatz von Bayern direkt abfragen
rank = r.zrevrank("standings", "FCB")   # 0-basiert, deswegen Addition in der nächsten Zeile
print(f"Bayern steht jetzt auf Platz {rank + 1}")
print(f"Neue Punkte: {new_pts}")

Bayern steht jetzt auf Platz 1
Neue Punkte: 34


---
## Schritt 8: TTL - Automatisches Ablaufen von Keys

Eine Killer-Funktion von Redis: Keys können mit Ablaufzeit gespeichert werden. Redis löscht sie automatisch.

### Use-Case: Live-Spielstand mit kurzer Lebensdauer

Wir cachen den aktuellen Spielstand für 5 Sekunden. So bekommt jeder User immer einen relativ aktuellen Stand, ohne dass wir bei jedem Aufruf die echte Datenquelle anfragen müssen.

In [27]:
r.set("livescore:match:1301", "Bayern 2:1 Dortmund (78. Min)", ex=5)

print("Wert:        ", r.get("livescore:match:1301"))
print("Restzeit (s):", r.ttl("livescore:match:1301"))
print("Warte 6 Sekunden...")

time.sleep(6)

print("Wert nach 6s:", r.get("livescore:match:1301"))   # None
print("TTL nach 6s :", r.ttl("livescore:match:1301"))   # -2 = Key existiert nicht

Warte 6 Sekunden...
Wert nach 6s: None
TTL nach 6s : -2


### TTL-Returncodes
- größer 0: Restzeit in Sekunden
- `-1`: Key existiert, hat aber keinen TTL (lebt unbegrenzt)
- `-2`: Key existiert nicht

In [28]:
r.set("permanent", "ich bleibe für immer")     # ohne ex
r.set("temporary", "ich gehe bald", ex=60)

print("permanent TTL: ", r.ttl("permanent"))         # -1
print("temporary TTL: ", r.ttl("temporary"))         # ~60
print("inexistent TTL:", r.ttl("does-not-exist"))    # -2

# TTL nachträglich entfernen (Key wird permanent)
r.persist("temporary")
print("temporary TTL nach PERSIST:", r.ttl("temporary"))   # -1

permanent TTL:  -1
temporary TTL:  60
inexistent TTL: -2
temporary TTL nach PERSIST: -1


---
## Schritt 9: User-Sessions mit Ablauf

Wenn sich ein User auf der Bundesliga-Seite einloggt, erzeugen wir ein Session-Token und speichern die zugehörigen Daten in Redis - mit TTL, damit alte Sessions automatisch verfallen.

### Login: Session anlegen

In [29]:
def login(user_id, username, favorite_team):
    token = secrets.token_urlsafe(16)
    key = f"session:{token}"
    r.hset(key, mapping={
        "user_id": user_id,
        "username": username,
        "favorite_team": favorite_team,
        "logged_in_at": datetime.now().isoformat()
    })
    # Session läuft nach 30 Minuten ab
    r.expire(key, 30 * 60)
    return token

session_token = login(42, "fcb_fan_2003", "FCB")
print(f"Login erfolgreich, Token: {session_token}")
print(f"Session läuft ab in: {r.ttl(f'session:{session_token}')} Sekunden")

Login erfolgreich, Token: YLEFoDbAPAkLcrMOeIpnsw
Session läuft ab in: 1800 Sekunden


### Folgeanfrage: Session validieren

In [ ]:
def get_session(token):
    key = f"session:{token}"
    data = r.hgetall(key)
    if not data:
        return None
    # Bei jedem Zugriff Session verlängern ('sliding expiration')
    r.expire(key, 30 * 60)
    return data

session = get_session(session_token)
print("Session-Inhalt:")
for k, v in session.items():
    print(f"  {k:15s} = {v}")

Session-Inhalt:
  user_id         = 42
  username        = fcb_fan_2003
  favorite_team   = FCB
  logged_in_at    = 2026-05-16T11:11:10.746495


### Logout: Session löschen

In [ ]:
def logout(token):
    return r.delete(f"session:{token}") == 1 # True -> Logout was successful

print("Vor Logout :", get_session(session_token))
logout(session_token)
print("Nach Logout:", get_session(session_token))

Vor Logout : {'user_id': '42', 'username': 'fcb_fan_2003', 'favorite_team': 'FCB', 'logged_in_at': '2026-05-16T11:11:10.746495'}
Nach Logout: None


---
## Schritt 10: Atomare Counter & Rate-Limiting

`INCR` erhöht einen Counter **atomar** - ohne Race-Conditions, ohne dass mehrere Clients sich in die Quere kommen.

### Zuschauer-Counter: "Wieviele schauen gerade Bayern-Dortmund?"

Für jeden View auf die Live-Seite zählen wir hoch. Bei Disconnects zählen wir runter.

In [30]:
r.delete("viewers:match:1301")

# 10.000 simultane Zuschauer kommen rein
for _ in range(10000):
    r.incr("viewers:match:1301")

print(f"Aktive Zuschauer: {r.get('viewers:match:1301')}")

# Ein paar Leute machen den Tab zu
for _ in range(347):
    r.decr("viewers:match:1301")

print(f"Nach Disconnects: {r.get('viewers:match:1301')}")

Aktive Zuschauer: 10000
Nach Disconnects: 9653


### Rate Limiting: max. 5 Requests pro Minute pro IP

Pattern: Counter pro IP, TTL = 60s. Wenn der Counter > 5, blocken.

In [ ]:
def rate_limit_ok(ip, max_requests=5, window_seconds=60):
    """
    Hier mit kleinem Limit (5) für die Demo, damit wir den Block sehen.
    In Produktion eher 60 oder 100 pro Minute.
    """
    key = f"rate:{ip}"
    count = r.incr(key)
    if count == 1:
        # Erster Request im Fenster -> TTL setzen
        r.expire(key, window_seconds)
    return count <= max_requests

# 7 Requests von derselben IP
print("Simuliere 7 Requests von 192.168.1.42:")
for i in range(1, 8):
    ok = rate_limit_ok("192.168.1.42")
    status = "OK     " if ok else "BLOCKED"
    print(f"  Request {i}: {status}  (counter={r.get('rate:192.168.1.42')})")

Simuliere 7 Requests von 192.168.1.42:
  Request 1: OK       (counter=1)
  Request 2: OK       (counter=2)
  Request 3: OK       (counter=3)
  Request 4: OK       (counter=4)
  Request 5: OK       (counter=5)
  Request 6: BLOCKED  (counter=6)
  Request 7: BLOCKED  (counter=7)


---
## Schritt 11: Replication - Master-Replica-Setup

Bisher haben wir mit einer einzelnen Redis-Instanz gearbeitet. Für ein Sportportal an einem Spitzenspiel-Samstag brauchen wir aber:
- **Ausfallsicherheit**: Wenn der Master in der 80. Minute crasht, übernimmt die Replica
- **Read-Skalierung**: Mehrere Replicas verteilen die Aufrufe auf die Spielstands-Seiten
- **Geografische Verteilung**: Replicas in verschiedenen Rechenzentren

### Topologie
```
            [ Master  :6379 ]
           /        |       \
   [ Replica :6380 ] ... [ Replica :6381 ]
```

Writes (z.B. neue Tor-Events) gehen immer an den Master, der die Änderungen asynchron an alle Replicas propagiert.

### Eine zweite Redis-Instanz als Replica starten

In Colab können wir einfach einen zweiten `redis-server` auf einem anderen Port starten. Mit der Option `--replicaof` macht er sich zum Slave des Masters auf 6379.

In [31]:
!redis-server --port 6380 --replicaof 127.0.0.1 6379 --daemonize yes --save "" --appendonly no
time.sleep(2)  # kurz warten bis Replica gestartet ist
print("Replica läuft auf Port 6380")

Replica läuft auf Port 6380


### Mit beiden Instanzen verbinden

In [32]:
master = redis.Redis(host='localhost', port=6379, decode_responses=True)
replica = redis.Redis(host='localhost', port=6380, decode_responses=True)

print("Master  PING:", master.ping())
print("Replica PING:", replica.ping())

Master  PING: True
Replica PING: True


### Replication-Status prüfen
`INFO replication` verrät uns die Rolle (master/slave) und Sync-Status.

In [33]:
def show_role(client, label):
    info = client.info("replication")
    print(f"--- {label} ---")
    print(f"  role:              {info['role']}")
    if info['role'] == 'master':
        print(f"  connected_slaves:  {info['connected_slaves']}")
        for k, v in info.items():
            if k.startswith("slave"):
                print(f"  {k}: {v}")
    else:
        print(f"  master_host:        {info.get('master_host')}")
        print(f"  master_port:        {info.get('master_port')}")
        print(f"  master_link_status: {info.get('master_link_status')}")

show_role(master, "MASTER (6379)") # offset = replication byte position "how far along in the replication stream"
print()
show_role(replica, "REPLICA (6380)")

--- MASTER (6379) ---
  role:              master
  connected_slaves:  1
  slave0: {'ip': '127.0.0.1', 'port': 6380, 'state': 'online', 'offset': 798, 'lag': 1}

--- REPLICA (6380) ---
  role:              slave
  master_host:        127.0.0.1
  master_port:        6379
  master_link_status: up


### Write am Master, Read von der Replica

In [34]:
master.set("breaking:news", "Tor! Bayern 3:1 Dortmund (Kane, 82.)")
time.sleep(0.1)   # kurze Propagationszeit abwarten

print("Master sieht :", master.get("breaking:news"))
print("Replica sieht:", replica.get("breaking:news"))

Master sieht : Tor! Bayern 3:1 Dortmund (Kane, 82.)
Replica sieht: Tor! Bayern 3:1 Dortmund (Kane, 82.)


### Write an die Replica -> Fehler

Replicas sind standardmäßig **read-only**. Versucht man trotzdem zu schreiben, gibt's einen Fehler. Das schützt vor versehentlichen Inkonsistenzen.

In [35]:
try:
    replica.set("forbidden", "geht nicht")
except redis.ReadOnlyError as e:
    print(f"Erwarteter Fehler: {e}")

Erwarteter Fehler: You can't write against a read only replica.


### Bulk-Sync testen: viele Writes am Master, dann von der Replica lesen

In [36]:
master.flushdb()
time.sleep(0.2)  # Replica muss FLUSHDB auch sehen

# 1000 Live-Updates auf den Master schreiben (Pipeline = batched, schnell)
pipe = master.pipeline()
for i in range(1000):
    pipe.set(f"event:{i}", f"Event-Nr-{i}")
pipe.execute()

time.sleep(0.3)  # Replication-Lag abwarten

print("Master-Keys  (gesamt):", master.dbsize())
print("Replica-Keys (gesamt):", replica.dbsize())
print("Beispiel-Read von Replica:", replica.get("event:42"))

Master-Keys  (gesamt): 1000
Replica-Keys (gesamt): 1000
Beispiel-Read von Replica: Event-Nr-42


### Frage an euch
Welche Konsistenzgarantien gibt Redis bei diesem Setup?
Was passiert, wenn der Master *direkt nach* einem Tor-Event crasht - bevor die Replica synchronisiert hat?

### Lösung
Redis-Replication ist **asynchron** → es gibt nur **Eventual Consistency**.

Wenn der Master nach einem `SET` crasht, *bevor* die Änderung die Replica erreicht hat:
- Die Replica hat den Write **nicht**
- Bei einem Failover (Replica wird zum neuen Master) ist der Write **verloren**

Für stärkere Garantien:
- `WAIT n timeout` - blockiert bis n Replicas den Write bestätigt haben
- **Redis Sentinel** - automatischer Failover mit Quorum
- **Redis Cluster** - Sharding + Replikation, integrierter Failover
- Für echte ACID-Konsistenz: Nicht Redis nutzen, sondern Postgres o.ä.

Für einen Liveticker ist das aber meistens OK: Wenn ein Tor-Event verloren geht, ist es schlimm - aber selten kriegt der Master crash genau in diesem 1-Millisekunden-Zeitfenster. Und die primäre DB hat den Event ja sowieso.

---
## Schritt 12: Partitioning & Scaling

Replication löst zwei Probleme:
- Ausfallsicherheit
- Read-Skalierung

**Nicht gelöst**: Write-Skalierung und RAM-Limits.
Ein einzelner Master kann nur so viele Writes verarbeiten wie eine Maschine schafft. Und alle Daten müssen in den RAM einer Maschine passen.

→ **Partitioning** (Sharding): Daten werden auf mehrere Master verteilt.

### Manuelles Client-Side-Sharding

Einfache Variante: Der Client entscheidet anhand des Keys, an welchen Shard er geht - z.B. per Hash modulo Shardanzahl.

In [ ]:
# Stelle dir vor wir hätten 3 Redis-Master
SHARDS = 3

def shard_for_key(key):
    h = int(hashlib.md5(key.encode()).hexdigest(), 16)
    return h % SHARDS

# Demo: zeigen, auf welchem Shard die Bundesliga-Vereine landen würden
print("Verteilung der Vereine auf 3 Shards:")
for code_ in TEAMS:
    print(f"  team:{code_:5s} -> Shard {shard_for_key('team:' + code_)}")

# Verteilung zählen
counts = Counter(shard_for_key("team:" + c) for c in TEAMS)
print(f"\nVerteilung gesamt: {dict(sorted(counts.items()))}")

Verteilung der Vereine auf 3 Shards:
  team:FCB   -> Shard 0
  team:BVB   -> Shard 1
  team:RBL   -> Shard 0
  team:B04   -> Shard 2
  team:VFB   -> Shard 0
  team:SGE   -> Shard 0
  team:TSG   -> Shard 0
  team:SCF   -> Shard 1
  team:FCA   -> Shard 2
  team:M05   -> Shard 1
  team:HSV   -> Shard 1
  team:FCU   -> Shard 1
  team:BMG   -> Shard 1
  team:KOE   -> Shard 1
  team:SVW   -> Shard 1
  team:WOB   -> Shard 2
  team:FCH   -> Shard 1
  team:STP   -> Shard 1

Verteilung gesamt: {0: 5, 1: 10, 2: 3}


### Probleme mit naivem Sharding
- **Resharding ist teuer**: Wenn man von 3 auf 4 Shards geht, ändert sich `hash % n` für fast alle Keys → fast alle Daten müssen umziehen
- **Hot-Keys**: Wenn ein Key sehr populär ist (`cache:match:match:1301` während des Topspiels), trifft die Last immer denselben Shard

**Lösungen:**
- **Consistent Hashing**: nur ca. 1/n der Keys ziehen beim Resharding um
- **Hash Slots** (so macht es Redis Cluster): 16384 feste Slots werden Servern zugeordnet

### Redis Cluster (Stichworte für die Folien)

Redis Cluster ist die native Sharding-Lösung:
- **16384 Hash Slots** werden auf die Master-Nodes verteilt
- Jeder Key landet in `CRC16(key) % 16384`
- Clients kennen die Slot-Verteilung und routen direkt
- Bei einem `MOVED`-Fehler erfährt der Client von einer Slot-Migration und folgt
- Pro Shard kann es Replicas geben → Failover automatisch
- **Keine Multi-Key-Ops über Shards hinweg** (außer mit Hash-Tags `{...}`)

### Frage an euch
Welche Topologie für den Bundesliga-Liveticker?

### Lösung
Für ein Sportportal in Deutschland:
- **1 Master + 2 Replicas** mit Sentinel reicht meistens
- Reads (Spielstandsabfragen) gehen an die Replicas - perfekte Lastverteilung am Samstagnachmittag
- Failover wird automatisch durch Sentinel gemanagt (kein 3-Uhr-morgens-On-Call-Anruf)
- Cache-Daten sind regenerierbar → kein Datenverlust-Drama bei Crash
- Tor-Events liegen sowieso in der primären DB persistent

Erst bei *richtig* großem Volumen (mehrere Ligen weltweit gleichzeitig, Millionen QPS) lohnt sich Redis Cluster mit Sharding. Für die deutsche Bundesliga allein: Overkill.

---
## Schritt 13: Shutdown

In [ ]:
# Beide Instanzen leeren
master.flushdb()

# Replica-Server stoppen
!redis-cli -p 6380 shutdown nosave 2>/dev/null
print("Replica gestoppt.")

# Master-Server stoppen
!redis-cli -p 6379 shutdown nosave 2>/dev/null
print("Master gestoppt.")

print("\nDemo zu Ende. Danke fürs Zuhören!")

Replica gestoppt.

Demo zu Ende. Danke fürs Zuhören!
